# 02 — Media Resolution & Captioning

Production-grade multimodal media ingestion pipeline for:
- image validation
- media quality analysis
- perceptual hashing
- duplicate detection
- caption generation
- caption confidence scoring
- artifact lineage generation

In [1]:
!pip install -q transformers accelerate sentencepiece
!pip install -q imagehash
!pip install -q opencv-python-headless
!pip install -q pyarrow fastparquet
!pip install -q sentence-transformers
!pip install -q faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 32.6 MB/s eta 0:00:00


In [2]:
import os
import gc
import cv2
import json
import time
import math
import torch
import random
import imagehash
import warnings
import numpy as np
import pandas as pd
import requests
import faiss
from io import BytesIO
from tqdm.auto import tqdm
from pathlib import Path
from dataclasses import dataclass
from PIL import Image

from concurrent.futures import ThreadPoolExecutor

from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration
)

from torchvision import transforms

from sentence_transformers import (
    SentenceTransformer
)

import torch.nn.functional as F

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE:", DEVICE)

if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

DEVICE: cuda
Tesla T4


In [3]:
@dataclass
class PipelineConfig:

    # PATHS
    INPUT_DATASET_PATH: str = "/kaggle/input/"
    WORKING_DIR: str = "/kaggle/working/"

    # ARTIFACTS
    ARTIFACT_DIR: str = "/kaggle/working/artifacts/v1/"

    # IMAGE SETTINGS
    IMAGE_SIZE: int = 224
    BATCH_SIZE: int = 32

    # CAPTION MODEL
    CAPTION_MODEL_NAME: str = "Salesforce/blip-image-captioning-base"

    # QUALITY THRESHOLDS
    MIN_RESOLUTION: int = 128
    BLUR_THRESHOLD: float = 40.0

CFG = PipelineConfig()

In [4]:
Path(CFG.ARTIFACT_DIR).mkdir(parents=True, exist_ok=True)

Path(f"{CFG.ARTIFACT_DIR}/media_cache").mkdir(
    parents=True,
    exist_ok=True
)

In [5]:
import glob

candidate_paths = glob.glob(
    "/kaggle/input/**/*.parquet",
    recursive=True
)

print("FOUND PARQUET FILES:\n")

for i, p in enumerate(candidate_paths):
    print(f"{i}: {p}")

FOUND PARQUET FILES:

0: /kaggle/input/notebooks/hanafudaearring/data-ingestion/ecomm_artifacts/products_df.parquet
1: /kaggle/input/notebooks/hanafudaearring/data-ingestion/ecomm_artifacts/reviews_df.parquet
2: /kaggle/input/notebooks/hanafudaearring/data-ingestion/ecomm_artifacts/product_level_df.parquet
3: /kaggle/input/notebooks/hanafudaearring/data-ingestion/ecomm_artifacts/media_df.parquet


In [6]:
TARGET_PARQUET = candidate_paths[0]

df = pd.read_parquet(TARGET_PARQUET)

print(df.shape)

df.head(2)

(728, 34)


,s.no,about_item,asin,availability,best_sellers_rank,brand_name,brand_page_url,breadcrumbs,customer_review_summary,default_variant/0,default_variant/1,default_variant/2,delivery_date,fastest_delivery_date,list_price,manufacturer,model_number,price_value,product_description,product_url,rating_count,rating_distribution/1star,rating_distribution/2star,rating_distribution/3star,rating_distribution/4star,rating_distribution/5star,rating_stars,recent_purchases,scrape_time,seller_name,seller_page_url,title,all_images,rank_1
0,0,Premium Comfort: Crafted from a high-quality c...,B0B59BJG6Y,In Stock,"#56,836 in Clothing, Shoes & Jewelry (See Top ...",MLYENX Store,https://www.amazon.com/stores/MLYENX/page/1FCD...,"Clothing, Shoes & Jewelry › Men › Clothing › A...",Customers find the shirts comfortable and well...,size:Large,"color:5 Pack Black, Dark Grey, Light Blue, Mil...",None,"Saturday, March 15","Tomorrow, March 11",List Price: $53.99,None,None,39.9926,None,https://www.amazon.com/dp/B0B59BJG6Y,"1,654 ratings",2%,1%,7%,15%,75%,4.6 out of 5 stars,50+ bought,03-10-2025 21:42,Greenfive,https://www.amazon.com/gp/help/seller/at-a-gla...,4/5 Pack Mens Polo Shirts Short Sleeve Quick D...,['https://m.media-amazon.com/images/I/41yUF65P...,85.0
1,1,Material: Men's polo shirt is made of soft pol...,B0DLGB4RYH,In Stock,"#50,261 in Clothing, Shoes & Jewelry (See Top ...",COOFANDY Store,https://www.amazon.com/stores/COOFANDY/page/23...,"Clothing, Shoes & Jewelry › Men › Clothing › S...",None,size:X-Large,color:Light Grey,None,"Saturday, March 15","Tomorrow, March 11",Typical price: $24.99,None,None,19.9920,None,https://www.amazon.com/dp/B0DLGB4RYH,20 ratings,0%,6%,11%,17%,66%,4.4 out of 5 stars,None,03-10-2025 21:42,COOFANDY,https://www.amazon.com/gp/help/seller/at-a-gla...,COOFANDY Men's Polo Shirts Short Sleeve Moistu...,['https://m.media-amazon.com/images/I/31a3mSs3...,199.0


In [7]:
print(df.columns.tolist())

['s.no', 'about_item', 'asin', 'availability', 'best_sellers_rank', 'brand_name', 'brand_page_url', 'breadcrumbs', 'customer_review_summary', 'default_variant/0', 'default_variant/1', 'default_variant/2', 'delivery_date', 'fastest_delivery_date', 'list_price', 'manufacturer', 'model_number', 'price_value', 'product_description', 'product_url', 'rating_count', 'rating_distribution/1star', 'rating_distribution/2star', 'rating_distribution/3star', 'rating_distribution/4star', 'rating_distribution/5star', 'rating_stars', 'recent_purchases', 'scrape_time', 'seller_name', 'seller_page_url', 'title', 'all_images', 'rank_1']


In [8]:
nulls = (
    df.isnull()
      .mean()
      .sort_values(ascending=False)
)

print(nulls)

default_variant/2            0.998626
model_number                 0.726648
manufacturer                 0.634615
product_description          0.627747
list_price                   0.449176
seller_page_url              0.414835
rank_1                       0.258242
best_sellers_rank            0.233516
recent_purchases             0.218407
customer_review_summary      0.129121
brand_page_url               0.108516
default_variant/1            0.096154
fastest_delivery_date        0.083791
default_variant/0            0.043956
delivery_date                0.035714
price_value                  0.028846
seller_name                  0.028846
breadcrumbs                  0.017857
availability                 0.017857
rating_count                 0.012363
rating_stars                 0.012363
s.no                         0.000000
about_item                   0.000000
asin                         0.000000
brand_name                   0.000000
product_url                  0.000000
rating_distr

In [9]:
possible_media_cols = [

    col for col in df.columns

    if any(
        keyword in col.lower()

        for keyword in [
            "image",
            "img",
            "media",
            "photo",
            "picture"
        ]
    )
]

print(possible_media_cols)

['all_images']


In [10]:
df["all_images"].iloc[0]

"['https://m.media-amazon.com/images/I/41yUF65PmbL.jpg', 'https://m.media-amazon.com/images/I/41jMweQcgzL.jpg', 'https://m.media-amazon.com/images/I/41QOPMJZtSL.jpg', 'https://m.media-amazon.com/images/I/5153qm1UTML.jpg', 'https://m.media-amazon.com/images/I/41JBALVmKiL.jpg', 'https://m.media-amazon.com/images/I/91eZ6+Gh9LL.SX38_SY50_CR,0,0,38,50_PKmb-play-button-overlay-thumb_.jpg']"

In [11]:
type(df["all_images"].iloc[0])

str

In [12]:
import ast

def parse_images(x):

    try:

        if isinstance(x, list):
            return x

        if isinstance(x, str):
            return ast.literal_eval(x)

        return []

    except:
        return []

In [13]:
df["parsed_images"] = df["all_images"].apply(parse_images)

In [14]:
df[[
    "asin",
    "parsed_images"
]].head()

,asin,parsed_images
0,B0B59BJG6Y,[https://m.media-amazon.com/images/I/41yUF65Pm...
1,B0DLGB4RYH,[https://m.media-amazon.com/images/I/31a3mSs3p...
2,B0DRXF62JH,[https://m.media-amazon.com/images/I/41J5q-Ua0...
3,B0DK5FZ325,[https://m.media-amazon.com/images/I/31J1ttvfy...
4,B0BGXTC1FR,[https://m.media-amazon.com/images/I/31Ff+3IbU...


In [15]:
image_df = df[[
    "asin",
    "title",
    "parsed_images"
]].explode("parsed_images")

In [16]:
image_df = image_df.rename(columns={
    "parsed_images": "image_path"
})

In [17]:
image_df = image_df[
    image_df["image_path"].notnull()
]

In [18]:
print("TOTAL PRODUCTS:", len(df))

print("TOTAL IMAGE RECORDS:", len(image_df))

print(
    "AVG IMAGES / PRODUCT:",
    round(len(image_df) / len(df), 2)
)

TOTAL PRODUCTS: 728
TOTAL IMAGE RECORDS: 4343
AVG IMAGES / PRODUCT: 5.97


In [19]:
image_df[[
    "asin",
    "image_path"
]].head(10)

,asin,image_path
0,B0B59BJG6Y,https://m.media-amazon.com/images/I/41yUF65Pmb...
0,B0B59BJG6Y,https://m.media-amazon.com/images/I/41jMweQcgz...
0,B0B59BJG6Y,https://m.media-amazon.com/images/I/41QOPMJZtS...
0,B0B59BJG6Y,https://m.media-amazon.com/images/I/5153qm1UTM...
0,B0B59BJG6Y,https://m.media-amazon.com/images/I/41JBALVmKi...
0,B0B59BJG6Y,https://m.media-amazon.com/images/I/91eZ6+Gh9L...
1,B0DLGB4RYH,https://m.media-amazon.com/images/I/31a3mSs3pa...
1,B0DLGB4RYH,https://m.media-amazon.com/images/I/41jqhQ7vML...
1,B0DLGB4RYH,https://m.media-amazon.com/images/I/41gWanpUu8...
1,B0DLGB4RYH,https://m.media-amazon.com/images/I/41l7ou7p0f...


In [20]:
sample_path = image_df["image_path"].iloc[0]

print(sample_path)
print(type(sample_path))

https://m.media-amazon.com/images/I/41yUF65PmbL.jpg
<class 'str'>


In [21]:
DOWNLOAD_DIR = (
    f"{CFG.ARTIFACT_DIR}/downloaded_images"
)

Path(DOWNLOAD_DIR).mkdir(
    parents=True,
    exist_ok=True
)

In [22]:
def download_image(url, save_path):

    try:

        response = requests.get(
            url,
            timeout=10
        )

        if response.status_code != 200:
            return False

        image = Image.open(
            BytesIO(response.content)
        ).convert("RGB")

        image.save(save_path)

        return True

    except:
        return False

In [23]:
image_df["local_image_path"] = [

    f"{DOWNLOAD_DIR}/{asin}_{idx}.jpg"

    for idx, asin in enumerate(image_df["asin"])
]

In [24]:
download_status = []

for _, row in tqdm(
    image_df.iterrows(),
    total=len(image_df)
):

    success = download_image(

        row["image_path"],

        row["local_image_path"]
    )

    download_status.append(success)

  0%|          | 0/4343 [00:00<?, ?it/s]

In [25]:
image_df["download_success"] = download_status

In [26]:
print(
    "SUCCESSFUL DOWNLOADS:",
    image_df["download_success"].sum()
)

print(
    "FAILED DOWNLOADS:",
    (~image_df["download_success"]).sum()
)

SUCCESSFUL DOWNLOADS: 4343
FAILED DOWNLOADS: 0


In [27]:
image_df = image_df[
    image_df["download_success"]
].reset_index(drop=True)

In [28]:
image_df["image_path"] = (
    image_df["local_image_path"]
)

In [29]:
def load_image(image_path):

    try:

        image = (
            Image.open(image_path)
            .convert("RGB")
        )

        return image

    except:
        return None

In [30]:
valid_records = []
failed_records = []

for _, row in tqdm(
    image_df.iterrows(),
    total=len(image_df)
):

    image = load_image(
        row["image_path"]
    )

    if image is None:

        failed_records.append({

            "asin": row["asin"],
            "image_path": row["image_path"]
        })

    else:

        valid_records.append({

            "asin": row["asin"],
            "image_path": row["image_path"]
        })

  0%|          | 0/4343 [00:00<?, ?it/s]

In [31]:
valid_images_df = pd.DataFrame(valid_records)

failed_images_df = pd.DataFrame(failed_records)

In [32]:
print(
    "VALID IMAGES:",
    len(valid_images_df)
)

print(
    "FAILED IMAGES:",
    len(failed_images_df)
)

VALID IMAGES: 4343
FAILED IMAGES: 0


In [33]:
metadata_records = []

for path in tqdm(
    valid_images_df["image_path"]
):

    try:

        image = Image.open(path)

        width, height = image.size

        metadata_records.append({

            "image_path": path,

            "width": width,

            "height": height,

            "aspect_ratio": width / height,

            "megapixels":
                (width * height) / 1e6
        })

    except:
        pass

  0%|          | 0/4343 [00:00<?, ?it/s]

In [34]:
media_metadata_df = pd.DataFrame(
    metadata_records
)

media_metadata_df.head()

,image_path,width,height,aspect_ratio,megapixels
0,/kaggle/working/artifacts/v1//downloaded_image...,400,500,0.8,0.20
1,/kaggle/working/artifacts/v1//downloaded_image...,500,500,1.0,0.25
2,/kaggle/working/artifacts/v1//downloaded_image...,500,500,1.0,0.25
3,/kaggle/working/artifacts/v1//downloaded_image...,500,500,1.0,0.25
4,/kaggle/working/artifacts/v1//downloaded_image...,500,500,1.0,0.25


In [35]:
print(
    media_metadata_df[[
        "width",
        "height",
        "megapixels"
    ]].describe()
)

             width       height   megapixels
count  4343.000000  4343.000000  4343.000000
mean    390.918490   441.220355     0.192993
std     151.324697   159.009442     0.092618
min      20.000000    20.000000     0.000400
25%     375.000000   500.000000     0.183500
50%     400.000000   500.000000     0.197500
75%     500.000000   500.000000     0.250000
max     993.000000   993.000000     0.986049


In [36]:
media_metadata_df["is_low_resolution"] = (

    (media_metadata_df["width"] < 300)
    |
    (media_metadata_df["height"] < 300)
)

In [37]:
media_metadata_df["is_extreme_aspect_ratio"] = (

    (media_metadata_df["aspect_ratio"] > 2.5)
    |
    (media_metadata_df["aspect_ratio"] < 0.4)
)

In [38]:
print(
    "LOW RESOLUTION IMAGES:",
    media_metadata_df["is_low_resolution"].sum()
)

print(
    "EXTREME ASPECT RATIO:",
    media_metadata_df[
        "is_extreme_aspect_ratio"
    ].sum()
)

LOW RESOLUTION IMAGES: 656
EXTREME ASPECT RATIO: 33


In [39]:
def compute_blur_score(image_path):

    try:

        image = cv2.imread(image_path)

        gray = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2GRAY
        )

        score = cv2.Laplacian(
            gray,
            cv2.CV_64F
        ).var()

        return score

    except:
        return np.nan

In [40]:
media_metadata_df["blur_score"] = [

    compute_blur_score(path)

    for path in tqdm(
        media_metadata_df["image_path"]
    )
]

  0%|          | 0/4343 [00:00<?, ?it/s]

In [41]:
media_metadata_df["blur_score"].describe()

count     4343.000000
mean      3424.457005
std       6026.468417
min          1.848120
25%        496.018854
50%       1085.257840
75%       3075.299965
max      80558.462735
Name: blur_score, dtype: float64

In [42]:
media_metadata_df["is_blurry"] = (

    media_metadata_df["blur_score"] < 40
)

In [43]:
print(
    "BLURRY IMAGES:",
    media_metadata_df["is_blurry"].sum()
)

BLURRY IMAGES: 28


In [44]:
def compute_brightness(image_path):

    try:

        image = cv2.imread(image_path)

        hsv = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2HSV
        )

        brightness = hsv[..., 2].mean()

        return brightness

    except:
        return np.nan

In [45]:
media_metadata_df["brightness_score"] = [

    compute_brightness(path)

    for path in tqdm(
        media_metadata_df["image_path"]
    )
]

  0%|          | 0/4343 [00:00<?, ?it/s]

In [46]:
media_metadata_df[
    "brightness_score"
].describe()

count    4343.000000
mean      181.467354
std        44.064637
min        19.521323
25%       153.957866
50%       192.096192
75%       214.466278
max       252.293812
Name: brightness_score, dtype: float64

In [47]:
media_metadata_df["is_dark_image"] = (

    media_metadata_df[
        "brightness_score"
    ] < 60
)

In [48]:
print(
    "DARK IMAGES:",
    media_metadata_df[
        "is_dark_image"
    ].sum()
)

DARK IMAGES: 54


In [49]:
def compute_contrast(image_path):

    try:

        image = cv2.imread(
            image_path,
            cv2.IMREAD_GRAYSCALE
        )

        return image.std()

    except:
        return np.nan

In [50]:
media_metadata_df["contrast_score"] = [

    compute_contrast(path)

    for path in tqdm(
        media_metadata_df["image_path"]
    )
]

  0%|          | 0/4343 [00:00<?, ?it/s]

In [51]:
media_metadata_df[
    "contrast_score"
].describe()

count    4343.000000
mean       69.223379
std        23.073222
min         1.898938
25%        53.690271
50%        70.247367
75%        86.590025
max       120.362642
Name: contrast_score, dtype: float64

In [52]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

quality_cols = [

    "megapixels",
    "blur_score",
    "brightness_score",
    "contrast_score"
]

media_metadata_df[
    quality_cols
] = scaler.fit_transform(

    media_metadata_df[quality_cols]
)

In [53]:
media_metadata_df["quality_score"] = (

    0.30 * media_metadata_df["megapixels"]
    +
    0.30 * media_metadata_df["blur_score"]
    +
    0.20 * media_metadata_df["brightness_score"]
    +
    0.20 * media_metadata_df["contrast_score"]
)

In [54]:
media_metadata_df[
    "quality_score"
].describe()

count    4343.000000
mean        0.324173
std         0.049574
min         0.067948
25%         0.295891
50%         0.328814
75%         0.358842
max         0.630664
Name: quality_score, dtype: float64

In [55]:
media_metadata_df["is_low_quality"] = (

    media_metadata_df[
        "quality_score"
    ] < 0.35
)

In [56]:
print(
    "LOW QUALITY IMAGES:",
    media_metadata_df[
        "is_low_quality"
    ].sum()
)

LOW QUALITY IMAGES: 2960


In [57]:
def compute_phash(image_path):

    try:

        image = Image.open(
            image_path
        ).convert("RGB")

        phash = imagehash.phash(image)

        return str(phash)

    except:
        return None

In [58]:
media_metadata_df["phash"] = [

    compute_phash(path)

    for path in tqdm(
        media_metadata_df["image_path"]
    )
]

  0%|          | 0/4343 [00:00<?, ?it/s]

In [59]:
media_metadata_df["phash"].nunique()

4156

In [60]:
duplicate_counts = (

    media_metadata_df["phash"]
    .value_counts()
    .reset_index()
)

duplicate_counts.columns = [
    "phash",
    "duplicate_count"
]

In [61]:
media_metadata_df = media_metadata_df.merge(

    duplicate_counts,

    on="phash",

    how="left"
)

In [62]:
media_metadata_df["is_duplicate"] = (

    media_metadata_df["duplicate_count"] > 1
)

In [63]:
print(
    "DUPLICATE IMAGES:",
    media_metadata_df[
        "is_duplicate"
    ].sum()
)

DUPLICATE IMAGES: 333


In [64]:
media_metadata_df[
    media_metadata_df["duplicate_count"] > 1
].sort_values(
    "duplicate_count",
    ascending=False
).head(20)

,image_path,width,height,aspect_ratio,megapixels,is_low_resolution,is_extreme_aspect_ratio,blur_score,is_blurry,brightness_score,is_dark_image,contrast_score,quality_score,is_low_quality,phash,duplicate_count,is_duplicate
4288,/kaggle/working/artifacts/v1//downloaded_image...,500,500,1.00,0.253234,False,False,0.037909,False,0.813716,False,0.663126,0.382711,False,ed5f574ca4a4a238,7,True
612,/kaggle/working/artifacts/v1//downloaded_image...,500,500,1.00,0.253234,False,False,0.037909,False,0.813716,False,0.663126,0.382711,False,ed5f574ca4a4a238,7,True
4253,/kaggle/working/artifacts/v1//downloaded_image...,500,500,1.00,0.253234,False,False,0.037909,False,0.813716,False,0.663126,0.382711,False,ed5f574ca4a4a238,7,True
4170,/kaggle/working/artifacts/v1//downloaded_image...,500,500,1.00,0.253234,False,False,0.148996,False,0.541162,False,0.912813,0.411464,False,bec0d0d0943e2f2f,7,True
4160,/kaggle/working/artifacts/v1//downloaded_image...,500,500,1.00,0.253234,False,False,0.148996,False,0.541162,False,0.912813,0.411464,False,bec0d0d0943e2f2f,7,True
4156,/kaggle/working/artifacts/v1//downloaded_image...,500,500,1.00,0.253234,False,False,0.148996,False,0.541162,False,0.912813,0.411464,False,bec0d0d0943e2f2f,7,True
4126,/kaggle/working/artifacts/v1//downloaded_image...,500,500,1.00,0.253234,False,False,0.145092,False,0.543323,False,0.912650,0.410692,False,bec0d0d0943e2f2f,7,True
4110,/kaggle/working/artifacts/v1//downloaded_image...,500,500,1.00,0.253234,False,False,0.148996,False,0.541162,False,0.912813,0.411464,False,bec0d0d0943e2f2f,7,True
4099,/kaggle/working/artifacts/v1//downloaded_image...,500,500,1.00,0.253234,False,False,0.148996,False,0.541162,False,0.912813,0.411464,False,bec0d0d0943e2f2f,7,True
4049,/kaggle/working/artifacts/v1//downloaded_image...,500,500,1.00,0.253234,False,False,0.145092,False,0.543323,False,0.912650,0.410692,False,bec0d0d0943e2f2f,7,True


In [65]:
IMAGE_SIZE = 224

transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[0.485, 0.456, 0.406],

        std=[0.229, 0.224, 0.225]
    )
])

In [66]:
NORMALIZED_DIR = (
    f"{CFG.ARTIFACT_DIR}/normalized_tensors"
)

Path(NORMALIZED_DIR).mkdir(
    parents=True,
    exist_ok=True
)

In [67]:
def normalize_image(image_path):

    try:

        image = (
            Image.open(image_path)
            .convert("RGB")
        )

        tensor = transform(image)

        return tensor

    except:
        return None

In [68]:
tensor_paths = []

for idx, row in tqdm(
    media_metadata_df.iterrows(),
    total=len(media_metadata_df)
):

    tensor = normalize_image(
        row["image_path"]
    )

    if tensor is None:

        tensor_paths.append(None)

        continue

    save_path = (
        f"{NORMALIZED_DIR}/"
        f"{idx}.pt"
    )

    torch.save(tensor, save_path)

    tensor_paths.append(save_path)

  0%|          | 0/4343 [00:00<?, ?it/s]

In [69]:
media_metadata_df["tensor_path"] = tensor_paths

In [70]:
print(
    "NORMALIZED TENSORS:",
    media_metadata_df[
        "tensor_path"
    ].notnull().sum()
)

NORMALIZED TENSORS: 4343


In [71]:
CAPTION_MODEL_NAME = (
    "Salesforce/blip-image-captioning-base"
)

processor = (
    BlipProcessor.from_pretrained(
        CAPTION_MODEL_NAME
    )
)

caption_model = (
    BlipForConditionalGeneration
    .from_pretrained(
        CAPTION_MODEL_NAME
    )
    .to(DEVICE)
)

caption_model.eval()

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

BlipForConditionalGeneration(
  (vision_model): BlipVisionModel(
    (embeddings): BlipVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (encoder): BlipEncoder(
      (layers): ModuleList(
        (0-11): 12 x BlipEncoderLayer(
          (self_attn): BlipAttention(
            (dropout): Dropout(p=0.0, inplace=False)
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (projection): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): BlipMLP(
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
      )
    )
    (post_layernorm): LayerNorm((768,), eps=1e-0

In [72]:
@torch.no_grad()
def generate_captions(batch_paths):

    images = []

    for path in batch_paths:

        image = (
            Image.open(path)
            .convert("RGB")
        )

        images.append(image)

    inputs = processor(

        images=images,

        return_tensors="pt",

        padding=True

    ).to(DEVICE)

    outputs = caption_model.generate(

        **inputs,

        max_new_tokens=30
    )

    captions = processor.batch_decode(

        outputs,

        skip_special_tokens=True
    )

    return captions

In [73]:
BATCH_SIZE = 16

caption_records = []

paths = media_metadata_df[
    "image_path"
].tolist()

for i in tqdm(

    range(0, len(paths), BATCH_SIZE)

):

    batch_paths = (
        paths[i:i+BATCH_SIZE]
    )

    try:

        captions = generate_captions(
            batch_paths
        )

        for path, caption in zip(
            batch_paths,
            captions
        ):

            caption_records.append({

                "image_path": path,

                "caption": caption
            })

    except Exception as e:

        print(e)

  0%|          | 0/272 [00:00<?, ?it/s]

In [74]:
captions_df = pd.DataFrame(
    caption_records
)

captions_df.head()

,image_path,caption
0,/kaggle/working/artifacts/v1//downloaded_image...,men ' s polo shirt
1,/kaggle/working/artifacts/v1//downloaded_image...,a man wearing a black polo shirt with a smile ...
2,/kaggle/working/artifacts/v1//downloaded_image...,a man wearing a grey polo shirt and white pants
3,/kaggle/working/artifacts/v1//downloaded_image...,a man walking with a golf bag
4,/kaggle/working/artifacts/v1//downloaded_image...,a man standing on the beach with his hands in ...


In [75]:
captions_df["caption"] = (

    captions_df["caption"]

    .str.strip()

    .str.lower()
)

In [76]:
captions_df["caption_length"] = (

    captions_df["caption"]

    .str.split()

    .apply(len)
)

In [77]:
captions_df["is_empty_caption"] = (

    captions_df["caption_length"] < 2
)

In [78]:
captions_df["is_repetitive_caption"] = (

    captions_df["caption"]

    .str.split()

    .apply(

        lambda x:
        len(set(x)) <= 2
    )
)

In [79]:
print(
    "EMPTY CAPTIONS:",
    captions_df[
        "is_empty_caption"
    ].sum()
)

print(
    "REPETITIVE CAPTIONS:",
    captions_df[
        "is_repetitive_caption"
    ].sum()
)

EMPTY CAPTIONS: 0
REPETITIVE CAPTIONS: 117


In [80]:
media_metadata_df = media_metadata_df.merge(

    captions_df,

    on="image_path",

    how="left"
)

In [81]:
media_metadata_df[[
    "image_path",
    "caption",
    "quality_score"
]].sample(10)

,image_path,caption,quality_score
564,/kaggle/working/artifacts/v1//downloaded_image...,a man wearing a brown sweater and jeans,0.359514
2452,/kaggle/working/artifacts/v1//downloaded_image...,sks sks sks sks sks sks sks sks sks sks sks sk...,0.426091
3331,/kaggle/working/artifacts/v1//downloaded_image...,a woman wearing a brown dress,0.348851
3672,/kaggle/working/artifacts/v1//downloaded_image...,the person in black,0.367453
2067,/kaggle/working/artifacts/v1//downloaded_image...,a man in a black boxer shorts,0.356642
4125,/kaggle/working/artifacts/v1//downloaded_image...,ge ge ge ge ge ge ge ge ge ge ge ge ge ge ge g...,0.395752
2058,/kaggle/working/artifacts/v1//downloaded_image...,"jockeys men ' s boxer shorts, navy, large",0.335690
3758,/kaggle/working/artifacts/v1//downloaded_image...,a brown silk sari with intricate motifs,0.119056
257,/kaggle/working/artifacts/v1//downloaded_image...,a man in a black t shirt and grey pants,0.347479
754,/kaggle/working/artifacts/v1//downloaded_image...,"men ' s jeans, men ' s jeans, men ' s jeans, m...",0.310300


In [82]:
EMBED_MODEL_NAME = (
    "clip-ViT-B-32"
)

clip_model = SentenceTransformer(
    EMBED_MODEL_NAME,
    device=DEVICE
)

modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: /root/.cache/huggingface/hub/models--sentence-transformers--clip-ViT-B-32/snapshots/327ab6726d33c0e22f920c83f2ff9e4bd38ca37f/0_CLIPModel
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [83]:
EMBED_BATCH_SIZE = 64

captions = media_metadata_df[
    "caption"
].fillna("").tolist()

len(captions)

4343

In [84]:
caption_embeddings = clip_model.encode(

    captions,

    batch_size=EMBED_BATCH_SIZE,

    show_progress_bar=True,

    convert_to_numpy=True,

    normalize_embeddings=True
)

Batches:   0%|          | 0/68 [00:00<?, ?it/s]

In [85]:
caption_embeddings.shape

(4343, 512)

In [86]:
image_paths = media_metadata_df[
    "image_path"
].tolist()

In [87]:
image_embeddings = clip_model.encode(

    image_paths,

    batch_size=32,

    show_progress_bar=True,

    convert_to_numpy=True,

    normalize_embeddings=True
)

Batches:   0%|          | 0/136 [00:00<?, ?it/s]

In [88]:
image_embeddings.shape

(4343, 512)

In [89]:
def cosine_similarity(a, b):

    return np.sum(a * b, axis=1)

In [90]:
alignment_scores = cosine_similarity(

    image_embeddings,

    caption_embeddings
)

In [91]:
media_metadata_df[
    "image_caption_similarity"
] = alignment_scores

In [92]:
media_metadata_df[
    "image_caption_similarity"
].describe()

count    4343.000000
mean        0.563847
std         0.127825
min         0.114647
25%         0.486813
50%         0.572338
75%         0.649745
max         0.874453
Name: image_caption_similarity, dtype: float64

In [93]:
media_metadata_df[
    "is_low_alignment"
] = (

    media_metadata_df[
        "image_caption_similarity"
    ] < 0.18
)

In [94]:
print(
    "LOW ALIGNMENT IMAGES:",
    media_metadata_df[
        "is_low_alignment"
    ].sum()
)

LOW ALIGNMENT IMAGES: 10


In [95]:
media_metadata_df[

    media_metadata_df[
        "is_low_alignment"
    ]

][[
    "image_path",
    "caption",
    "quality_score",
    "image_caption_similarity"
]].head(20)

,image_path,caption,quality_score,image_caption_similarity
833,/kaggle/working/artifacts/v1//downloaded_image...,a man in a plaid shirt and jeans standing in f...,0.223981,0.162698
2496,/kaggle/working/artifacts/v1//downloaded_image...,the nike react react is a grey and white shoe,0.254898,0.175453
2693,/kaggle/working/artifacts/v1//downloaded_image...,a man in a black suit and a white shirt standi...,0.310856,0.168082
3134,/kaggle/working/artifacts/v1//downloaded_image...,a woman sitting on a white background wearing ...,0.326411,0.130883
3740,/kaggle/working/artifacts/v1//downloaded_image...,a woman wearing a beige sari with floral embro...,0.314613,0.165366
3763,/kaggle/working/artifacts/v1//downloaded_image...,a woman wearing a lavender colored sala suit w...,0.228478,0.136412
3771,/kaggle/working/artifacts/v1//downloaded_image...,a white embroidered tablecloth with pink flowe...,0.314720,0.168675
3819,/kaggle/working/artifacts/v1//downloaded_image...,a woman in a black sari with gold detailing an...,0.196662,0.114647
3921,/kaggle/working/artifacts/v1//downloaded_image...,fruit & amp boys ' 3 - pack long sleeve t - sh...,0.344879,0.175410
4208,/kaggle/working/artifacts/v1//downloaded_image...,a man sitting on a bench wearing a maroon polo...,0.276256,0.162855


In [96]:
fusion_embeddings = (

    0.6 * image_embeddings
    +
    0.4 * caption_embeddings
)

In [97]:
fusion_embeddings = (

    fusion_embeddings
    /
    np.linalg.norm(
        fusion_embeddings,
        axis=1,
        keepdims=True
    )
)

fusion_embeddings.shape

(4343, 512)

In [98]:
embedding_dim = (
    fusion_embeddings.shape[1]
)

embedding_dim

512

In [99]:
index = faiss.IndexFlatIP(
    embedding_dim
)

In [100]:
index.add(

    fusion_embeddings.astype(
        np.float32
    )
)

In [101]:
index.ntotal

4343

In [102]:
def search_by_text(

    query,
    top_k=5
):

    query_embedding = clip_model.encode(

        [query],

        normalize_embeddings=True
    )

    scores, indices = index.search(

        query_embedding.astype(
            np.float32
        ),

        top_k
    )

    results = media_metadata_df.iloc[
        indices[0]
    ][[
        "image_path",
        "caption",
        "quality_score"
    ]]

    results["similarity_score"] = (
        scores[0]
    )

    return results

In [103]:
search_by_text(

    "red floral dress",

    top_k=10
)

,image_path,caption,quality_score,similarity_score
3269,/kaggle/working/artifacts/v1//downloaded_image...,a woman wearing a red floral dress,0.319141,0.668943
3270,/kaggle/working/artifacts/v1//downloaded_image...,a woman wearing a red floral dress,0.320007,0.668906
3268,/kaggle/working/artifacts/v1//downloaded_image...,a woman wearing a red floral dress,0.278640,0.668662
4142,/kaggle/working/artifacts/v1//downloaded_image...,a baby girl in a red sweater and floral skirt,0.336024,0.628048
2782,/kaggle/working/artifacts/v1//downloaded_image...,a woman wearing a red floral shirt,0.347993,0.618834
2798,/kaggle/working/artifacts/v1//downloaded_image...,a woman wearing a red floral top with a white ...,0.330960,0.615765
3265,/kaggle/working/artifacts/v1//downloaded_image...,a woman wearing a red dress with white flowers,0.345998,0.609552
3162,/kaggle/working/artifacts/v1//downloaded_image...,a woman wearing a red floral dress and hat,0.364937,0.602453
3286,/kaggle/working/artifacts/v1//downloaded_image...,a woman wearing a floral print dress,0.333910,0.587871
3178,/kaggle/working/artifacts/v1//downloaded_image...,a woman wearing a floral print dress,0.364697,0.583874


In [104]:
def search_by_image(

    image_idx,
    top_k=5
):

    query_embedding = fusion_embeddings[
        image_idx
    ].reshape(1, -1)

    scores, indices = index.search(

        query_embedding.astype(
            np.float32
        ),

        top_k
    )

    results = media_metadata_df.iloc[
        indices[0]
    ][[
        "image_path",
        "caption",
        "quality_score"
    ]]

    results["similarity_score"] = (
        scores[0]
    )

    return results

In [105]:
search_by_image(

    image_idx=100,

    top_k=10
)

,image_path,caption,quality_score,similarity_score
100,/kaggle/working/artifacts/v1//downloaded_image...,the north face men ' s short sleeve crew neck ...,0.339006,1.000000
3979,/kaggle/working/artifacts/v1//downloaded_image...,the north face men ' s short sleeve crew neck ...,0.411260,0.994776
328,/kaggle/working/artifacts/v1//downloaded_image...,the north face men ' s short sleeve t shirt,0.375737,0.991438
358,/kaggle/working/artifacts/v1//downloaded_image...,the north face men ' s short sleeve t shirt,0.312174,0.990603
1798,/kaggle/working/artifacts/v1//downloaded_image...,the north face men ' s short sleeve t shirt,0.312174,0.990218
3945,/kaggle/working/artifacts/v1//downloaded_image...,the north face men ' s short sleeve t shirt,0.212298,0.989877
1910,/kaggle/working/artifacts/v1//downloaded_image...,the north face men ' s short sleeve t shirt,0.375373,0.982558
538,/kaggle/working/artifacts/v1//downloaded_image...,the north face men ' s long sleeve t shirt,0.314046,0.979585
537,/kaggle/working/artifacts/v1//downloaded_image...,the north face men ' s meral long sleeve t shirt,0.325728,0.979561
1463,/kaggle/working/artifacts/v1//downloaded_image...,the north face men ' s horizon short sleeve t ...,0.338440,0.978857


In [106]:
duplicate_examples = media_metadata_df[

    media_metadata_df[
        "duplicate_count"
    ] > 1
]

duplicate_examples.head()

,image_path,width,height,aspect_ratio,megapixels,is_low_resolution,is_extreme_aspect_ratio,blur_score,is_blurry,brightness_score,is_dark_image,contrast_score,quality_score,is_low_quality,phash,duplicate_count,is_duplicate,tensor_path,caption,caption_length,is_empty_caption,is_repetitive_caption,image_caption_similarity,is_low_alignment
13,/kaggle/working/artifacts/v1//downloaded_image...,421,500,0.842,0.213159,False,False,0.015027,False,0.495026,False,0.827517,0.332964,True,ba62c5ac8a1dadb8,2,True,/kaggle/working/artifacts/v1//normalized_tenso...,men ' s polo shirt short sleeved polo shirt,9,False,False,0.604274,False
16,/kaggle/working/artifacts/v1//downloaded_image...,421,500,0.842,0.213159,False,False,0.011925,False,0.182293,False,0.500163,0.204016,True,cd4db03434b633d3,2,True,/kaggle/working/artifacts/v1//normalized_tenso...,a logo for a fire company,6,False,False,0.624209,False
17,/kaggle/working/artifacts/v1//downloaded_image...,421,500,0.842,0.213159,False,False,0.022021,False,0.590834,False,0.592877,0.307296,True,ac931368d43bb44f,2,True,/kaggle/working/artifacts/v1//normalized_tenso...,"a man with a beard and a beard, wearing differ...",14,False,False,0.390924,False
18,/kaggle/working/artifacts/v1//downloaded_image...,38,50,0.760,0.001522,True,False,0.152335,False,0.784738,False,0.607484,0.324602,True,fca2a14c5ab2656d,2,True,/kaggle/working/artifacts/v1//normalized_tenso...,a black and white diamond ring,6,False,False,0.539338,False
33,/kaggle/working/artifacts/v1//downloaded_image...,382,500,0.764,0.193375,False,False,0.000774,False,0.990181,False,0.059163,0.268114,True,bc4bc2e08f1e4b72,2,True,/kaggle/working/artifacts/v1//normalized_tenso...,a white t shirt with a black logo on the front,11,False,False,0.618528,False


In [107]:
test_idx = duplicate_examples.index[0]

search_by_image(
    test_idx,
    top_k=10
)

,image_path,caption,quality_score,similarity_score
13,/kaggle/working/artifacts/v1//downloaded_image...,men ' s polo shirt short sleeved polo shirt,0.332964,1.000000
3372,/kaggle/working/artifacts/v1//downloaded_image...,men ' s polo shirt,0.336043,0.984231
52,/kaggle/working/artifacts/v1//downloaded_image...,men ' s polo shirt,0.350889,0.984183
42,/kaggle/working/artifacts/v1//downloaded_image...,men ' s golf polo shirt,0.343144,0.983863
0,/kaggle/working/artifacts/v1//downloaded_image...,men ' s polo shirt,0.326877,0.983805
3391,/kaggle/working/artifacts/v1//downloaded_image...,men ' s polo shirt,0.361830,0.982872
120,/kaggle/working/artifacts/v1//downloaded_image...,men ' s golf polo shirt,0.366996,0.982797
124,/kaggle/working/artifacts/v1//downloaded_image...,men ' s polo shirt,0.357704,0.982243
80,/kaggle/working/artifacts/v1//downloaded_image...,3 pack of men ' s polo shirts,0.326709,0.978647
278,/kaggle/working/artifacts/v1//downloaded_image...,the men ' s polo shirt with collar and cuffs,0.375074,0.978429


In [108]:
np.save(

    f"{CFG.ARTIFACT_DIR}/fusion_embeddings.npy",

    fusion_embeddings
)

In [109]:
np.save(

    f"{CFG.ARTIFACT_DIR}/caption_embeddings.npy",

    caption_embeddings
)

In [110]:
faiss.write_index(

    index,

    f"{CFG.ARTIFACT_DIR}/multimodal_faiss.index"
)

In [111]:
media_metadata_df.to_parquet(

    f"{CFG.ARTIFACT_DIR}/retrieval_ready_df.parquet",

    index=False
)

In [112]:
print("=" * 60)

print("MULTIMODAL MEDIA PIPELINE COMPLETE")

print("=" * 60)

print(
    "TOTAL MEDIA RECORDS:",
    len(media_metadata_df)
)

print(
    "FAISS INDEX SIZE:",
    index.ntotal
)

print(
    "DUPLICATE IMAGES:",
    media_metadata_df[
        "is_duplicate"
    ].sum()
)

print(
    "LOW QUALITY IMAGES:",
    media_metadata_df[
        "is_low_quality"
    ].sum()
)

print(
    "LOW ALIGNMENT IMAGES:",
    media_metadata_df[
        "is_low_alignment"
    ].sum()
)

print("=" * 60)

MULTIMODAL MEDIA PIPELINE COMPLETE
TOTAL MEDIA RECORDS: 4343
FAISS INDEX SIZE: 4343
DUPLICATE IMAGES: 333
LOW QUALITY IMAGES: 2960
LOW ALIGNMENT IMAGES: 10
